In [1]:
import ir_datasets
dataset = ir_datasets.load("wikir/en1k/training")
doc_generator = (doc.text for doc in dataset.docs_iter())
print("docs generator created!")

docs generator created!


In [2]:
from nltk.stem import PorterStemmer

porter_stemmer = PorterStemmer()
stemmed_words = [porter_stemmer.stem(word) for word in doc_generator]
print("stemmed words ready!")

stemmed words ready!


### TfidfVectorizer

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf_idf_vect = TfidfVectorizer(stop_words="english")
tf_idf_matrix = tf_idf_vect.fit_transform(stemmed_words)
print("TF-IDF matrix ready!")

TF-IDF matrix ready!


In [4]:
queries = [query.text for query in dataset.queries_iter()]

In [ ]:
query_vectors = tf_idf_vect.transform(queries)

In [7]:
from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(query_vectors, tf_idf_matrix)

- necessary dictionaries

In [8]:
from collections import defaultdict

doc_dict = defaultdict(str)

for i, doc in enumerate(dataset.docs_iter()):
    doc_dict[i] = doc.doc_id

doc_dict = dict(doc_dict)

In [9]:
qrels_dict = defaultdict(list)

for qrel in dataset.qrels_iter():
    qrels_dict[qrel.query_id].append(qrel.doc_id)

qrels_dict = dict(qrels_dict)

In [10]:
from helper import Scoredoc

score_doc_dict = defaultdict(list)

for scoreddoc in dataset.scoreddocs_iter():
    doc_id = scoreddoc.doc_id
    score = scoreddoc.score

    scoreddoc_object = Scoredoc(doc_id, score)

    score_doc_dict[scoreddoc.query_id].append(scoreddoc_object)

score_doc_dict = dict(score_doc_dict)

In [11]:
import pandas as pd
query_ids = [query.query_id for query in dataset.queries_iter()]
df = pd.DataFrame(query_ids, columns=["Query_ID"])
df

,Query_ID
0,123839
1,188629
2,13898
3,316959
4,515031
...,...
1439,896124
1440,12319
1441,4421
1442,296526


In [13]:
from helper import create_AP, create_ndcg, create_statistical_columns, print_columns
df = create_statistical_columns(df, qrels_dict, doc_dict, similarities)
df = create_AP(df, qrels_dict, doc_dict, similarities)
df = create_ndcg(df, doc_dict, similarities, score_doc_dict)

print_columns(df)

recall_5_mean: 12.489096285847333
recall_5_std: 13.739339789661496
recall_5_max: 83.33333333333334
recall_5_min: 0.0
recall_10_mean: 17.380854250971783
recall_10_std: 18.131309617109874
recall_10_max: 100.0
recall_10_min: 0.0
precision_5_mean: 26.05263157894737
precision_5_std: 22.552011187324478
precision_5_max: 100.0
precision_5_min: 0.0
precision_10_mean: 19.037396121883656
precision_10_std: 16.912845372871633
precision_10_max: 90.0
precision_10_min: 0.0
f_score_5_mean: 15.624484554584486
f_score_5_std: 15.928877895207622
f_score_5_max: 90.9090909090909
f_score_5_min: 0.0
f_score_10_mean: 16.32223459155353
f_score_10_std: 15.525480224320377
f_score_10_max: 88.88888888888889
f_score_10_min: 0.0
MAP_5: 0.097352327020423
MAP_10: 0.11742378009414076
NDCG_5_mean: 0.47878261809755285
NDCG_5_std: 0.34592004765318385
NDCG_5_max: 1.0000000000000002
NDCG_5_min: 0.0
NDCG_10_mean: 0.47932204006982687
NDCG_10_std: 0.33979386832001757
NDCG_10_max: 1.0000000000000002
NDCG_10_min: 0.0
